setup + data

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path().resolve().parent / "src"))

import numpy as np

from config import get_settings
from data.storage import read_parquet
from features.pipeline import prepare_training_frame
from models.backtest import walk_forward_backtest
from models.train import train_model, FEATURES

s = get_settings()

processed = sorted((s.data_dir / "processed").glob("matches_t*.parquet"))
df = read_parquet(processed[-1])

df_ml = prepare_training_frame(df, form_window=5)
df_ml.shape

backtest

In [ ]:
metrics = walk_forward_backtest(df_ml, n_splits=5)
{k: v for k, v in metrics.items() if k != "oof_proba"}

train final model + save

In [ ]:
from models.registry import save_model
from data.storage import write_parquet

model = train_model(df_ml)
save_model(s.data_dir / "reports" / "model.joblib", model)

df_ml["oof_proba"] = metrics["oof_proba"]
write_parquet(s.data_dir / "reports" / "train_frame.parquet", df_ml)

print("Saved:", s.data_dir / "reports" / "model.joblib")
print("Saved:", s.data_dir / "reports" / "train_frame.parquet")